In [ ]:
!pip install datasets jiwer albumentations torchmetrics -q

In [ ]:
import os, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import OneCycleLR
import torchvision.transforms as T
from PIL import Image
from datasets import load_dataset
from jiwer import cer, wer
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Using device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## ⚙️ Cell 3 — Hyperparameters (Edit Here)

In [ ]:
# ── Image ──────────────────────────────────────────────────────
IMG_H       = 64        # Fixed height for all images
IMG_W       = 1024      # Fixed width  (stretch resize)

# ── Model ──────────────────────────────────────────────────────
D_MODEL     = 256       # Transformer hidden dim  (128 for free T4, 256 for A100)
NHEAD       = 8         # Attention heads         (must divide D_MODEL)
NUM_LAYERS  = 4         # Transformer encoder layers
DROPOUT     = 0.1

# ── Training ───────────────────────────────────────────────────
BATCH       = 16        # Reduce to 8 if OOM on T4
EPOCHS      = 60
LR          = 3e-4
WEIGHT_DECAY= 1e-4
GRAD_CLIP   = 5.0
NUM_WORKERS = 2

# ── Checkpointing ──────────────────────────────────────────────
SAVE_DIR    = '/content/drive/MyDrive/IAMline_HTR'   # change if not using Drive
SAVE_EVERY  = 5         # Save checkpoint every N epochs

print('✅ Hyperparameters set.')

In [ ]:
USE_DRIVE = True   # Set False to skip

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f'✅ Drive mounted. Checkpoints → {SAVE_DIR}')
else:
    SAVE_DIR = '/content/checkpoints'
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f'⚠️  Drive skipped. Saving locally to {SAVE_DIR} (lost on disconnect!)')

In [ ]:
print('Loading Teklia/IAMline from HuggingFace...')
dataset = load_dataset('Teklia/IAM-line')
print(dataset)
print(f"\nTrain size : {len(dataset['train'])}")
print(f"Val size   : {len(dataset['validation'])}")
print(f"Test size  : {len(dataset['test'])}")

# Preview a sample
sample = dataset['train'][0]
print(f"\nSample text : {sample['text']}")
print(f"Image size  : {sample['image'].size}")
sample['image']

In [ ]:
import string

# Build vocab from actual dataset characters (safer than hardcoding)
all_texts = (
    [s['text'] for s in dataset['train']] +
    [s['text'] for s in dataset['validation']] +
    [s['text'] for s in dataset['test']]
)
unique_chars = sorted(set(''.join(all_texts)))

VOCAB    = ['<blank>'] + unique_chars   # index 0 = CTC blank
char2idx = {c: i for i, c in enumerate(VOCAB)}
idx2char = {i: c for c, i in char2idx.items()}
VOCAB_SIZE = len(VOCAB)

print(f'Vocabulary size : {VOCAB_SIZE}')
print(f'Characters      : {"".join(unique_chars)}')

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── Transforms ────────────────────────────────────────────────
def get_train_transform():
    return T.Compose([
        T.Grayscale(),
        T.Resize((IMG_H, IMG_W)),
        T.RandomApply([T.GaussianBlur(3, sigma=(0.1, 1.0))], p=0.3),
        T.RandomApply([T.ColorJitter(brightness=0.3, contrast=0.3)], p=0.4),
        T.RandomAffine(degrees=2, translate=(0.02, 0.02), scale=(0.95, 1.05)),
        T.ToTensor(),
        T.Normalize(mean=[0.5], std=[0.5])
    ])

def get_val_transform():
    return T.Compose([
        T.Grayscale(),
        T.Resize((IMG_H, IMG_W)),
        T.ToTensor(),
        T.Normalize(mean=[0.5], std=[0.5])
    ])

# ── Dataset ───────────────────────────────────────────────────
class IAMDataset(Dataset):
    def __init__(self, hf_split, transform):
        self.data      = hf_split
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item  = self.data[idx]
        img   = item['image'].convert('RGB')   # PIL
        img   = self.transform(img)             # (1, H, W)
        text  = item['text']
        label = torch.tensor(
            [char2idx[c] for c in text if c in char2idx],
            dtype=torch.long
        )
        return img, label, text

# ── Collate ───────────────────────────────────────────────────
def collate_fn(batch):
    imgs, labels, texts = zip(*batch)
    imgs         = torch.stack(imgs)                      # (B, 1, H, W)
    label_lens   = torch.tensor([len(l) for l in labels], dtype=torch.long)
    labels_cat   = torch.cat(labels)                      # flat for CTC
    return imgs, labels_cat, label_lens, texts

# ── DataLoaders ───────────────────────────────────────────────
train_ds = IAMDataset(dataset['train'],      get_train_transform())
val_ds   = IAMDataset(dataset['validation'], get_val_transform())
test_ds  = IAMDataset(dataset['test'],       get_val_transform())

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                      collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                      collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False,
                      collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches : {len(train_dl)}')
print(f'Val   batches : {len(val_dl)}')
print(f'Test  batches : {len(test_dl)}')

In [ ]:
# ── Residual Conv Block ────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU()
        )
        self.skip = (
            nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False)
            if in_ch != out_ch or stride != 1 else nn.Identity()
        )

    def forward(self, x):
        return self.block(x) + self.skip(x)

# ── CNN Backbone ──────────────────────────────────────────────
class CNNBackbone(nn.Module):
    """
    Input : (B, 1, 64, 1024)
    Output: (T, B, d_model)   where T ≈ W/4
    """
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            ConvBlock(1,  32),
            nn.MaxPool2d(2, 2),              # → (B, 32,  32, W/2)
            ConvBlock(32, 64),
            nn.MaxPool2d(2, 2),              # → (B, 64,  16, W/4)
            ConvBlock(64, 128),
            nn.MaxPool2d((2,1), (2,1)),      # → (B, 128,  8, W/4)
            ConvBlock(128, 256),
            nn.MaxPool2d((2,1), (2,1)),      # → (B, 256,  4, W/4)
            ConvBlock(256, d_model),
            nn.AdaptiveAvgPool2d((1, None))  # → (B, d_model, 1, T)
        )

    def forward(self, x):
        x = self.net(x)         # (B, d_model, 1, T)
        x = x.squeeze(2)        # (B, d_model, T)
        return x.permute(2,0,1) # (T, B, d_model)

# ── Sinusoidal Positional Encoding ────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(1))  # (max_len, 1, d_model)

    def forward(self, x):   # x: (T, B, d_model)
        return self.dropout(x + self.pe[:x.size(0)])

# ── Full HTR Model ────────────────────────────────────────────
class HTRModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4, dropout=0.1):
        super().__init__()
        self.cnn     = CNNBackbone(d_model)
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)
        enc_layer    = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=False,
            norm_first=True    # Pre-LayerNorm = more stable
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head        = nn.Linear(d_model, vocab_size)

    def forward(self, x):          # x: (B, 1, H, W)
        x = self.cnn(x)            # (T, B, d_model)
        x = self.pos_enc(x)
        x = self.transformer(x)    # (T, B, d_model)
        return self.head(x)        # (T, B, vocab_size)

# ── Instantiate ───────────────────────────────────────────────
model = HTRModel(
    vocab_size = VOCAB_SIZE,
    d_model    = D_MODEL,
    nhead      = NHEAD,
    num_layers = NUM_LAYERS,
    dropout    = DROPOUT
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model built — {total_params/1e6:.2f}M trainable parameters')

# Quick shape check
with torch.no_grad():
    dummy = torch.randn(2, 1, IMG_H, IMG_W).to(DEVICE)
    out   = model(dummy)
    print(f'   Input shape  : {list(dummy.shape)}')
    print(f'   Output shape : {list(out.shape)}  (T, B, vocab_size)')

In [ ]:
ctc_loss  = nn.CTCLoss(blank=0, zero_infinity=True).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = OneCycleLR(
    optimizer,
    max_lr           = LR,
    steps_per_epoch  = len(train_dl),
    epochs           = EPOCHS,
    pct_start        = 0.1,       # 10% warmup
    anneal_strategy  = 'cos'
)

print('✅ CTC loss / AdamW / OneCycleLR ready.')

In [ ]:
def greedy_decode(logits):
    """
    logits : (T, B, V)
    returns: list of decoded strings, length B
    """
    indices = logits.argmax(2).permute(1, 0)   # (B, T)
    results = []
    for seq in indices.cpu().tolist():
        chars, prev = [], -1
        for idx in seq:
            if idx != prev and idx != 0:        # skip blank & repeats
                chars.append(idx2char.get(idx, ''))
            prev = idx
        results.append(''.join(chars))
    return results

print('✅ Greedy CTC decoder ready.')

In [ ]:
from torch.cuda.amp import GradScaler, autocast

scaler = GradScaler()   # Mixed-precision for speed

# ── History ───────────────────────────────────────────────────
history = {'train_loss': [], 'val_cer': [], 'val_wer': []}
best_cer = float('inf')

# ── Train one epoch ───────────────────────────────────────────
def train_epoch():
    model.train()
    total_loss = 0.0
    for imgs, labels, label_lens, _ in train_dl:
        imgs      = imgs.to(DEVICE)
        labels    = labels.to(DEVICE)
        label_lens= label_lens.to(DEVICE)

        with autocast():
            logits     = model(imgs)                             # (T, B, V)
            log_probs  = logits.log_softmax(2)
            input_lens = torch.full(
                (imgs.size(0),), logits.size(0),
                dtype=torch.long, device=DEVICE
            )
            loss = ctc_loss(log_probs, labels, input_lens, label_lens)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(train_dl)

# ── Evaluate ──────────────────────────────────────────────────
@torch.no_grad()
def evaluate(loader):
    model.eval()
    all_preds, all_targets = [], []
    for imgs, labels, label_lens, texts in loader:
        imgs   = imgs.to(DEVICE)
        logits = model(imgs)
        preds  = greedy_decode(logits)
        all_preds.extend(preds)
        all_targets.extend(texts)
    c = cer(all_targets, all_preds)
    w = wer(all_targets, all_preds)
    return round(c, 4), round(w, 4), all_preds, all_targets

# ── Save checkpoint ───────────────────────────────────────────
def save_checkpoint(epoch, val_cer, tag='best'):
    path = os.path.join(SAVE_DIR, f'htr_{tag}.pt')
    torch.save({
        'epoch'     : epoch,
        'model'     : model.state_dict(),
        'optimizer' : optimizer.state_dict(),
        'val_cer'   : val_cer,
        'vocab'     : VOCAB,
        'char2idx'  : char2idx,
    }, path)
    print(f'   💾 Saved {tag} checkpoint → {path}')

print('✅ Training functions ready.')

In [ ]:
import torch
import os
# Load best checkpoint
ckpt = torch.load(os.path.join(SAVE_DIR, 'htr_best.pt'), map_location=DEVICE)
model.load_state_dict(ckpt['model'])

In [ ]:
print(f'🚀 Starting training for {EPOCHS} epochs on {DEVICE}\n')
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Val CER":>8} | {"Val WER":>8} | {"Status"}')
print('-' * 56)

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch()
    val_cer_v, val_wer_v, _, _ = evaluate(val_dl)

    history['train_loss'].append(train_loss)
    history['val_cer'].append(val_cer_v)
    history['val_wer'].append(val_wer_v)

    status = ''
    if val_cer_v < best_cer:
        best_cer = val_cer_v
        save_checkpoint(epoch, val_cer_v, tag='best')
        status = '🏆 NEW BEST'

    if epoch % SAVE_EVERY == 0:
        save_checkpoint(epoch, val_cer_v, tag=f'epoch{epoch:03d}')

    print(f'{epoch:>6} | {train_loss:>10.4f} | {val_cer_v:>8.4f} | {val_wer_v:>8.4f} | {status}')

print(f'\n✅ Training complete. Best Val CER: {best_cer:.4f}')

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(epochs_range, history['train_loss'], color='steelblue', linewidth=2)
ax1.set_title('Training Loss (CTC)', fontsize=13)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(alpha=0.3)

ax2.plot(epochs_range, history['val_cer'], label='CER', color='tomato', linewidth=2)
ax2.plot(epochs_range, history['val_wer'], label='WER', color='darkorange', linewidth=2, linestyle='--')
ax2.set_title('Validation CER / WER', fontsize=13)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Error Rate')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(SAVE_DIR, 'training_curves.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f'Plot saved to {plot_path}')

In [ ]:
import torch
import os
# Load best checkpoint
ckpt = torch.load(os.path.join(SAVE_DIR, 'htr_best.pt'), map_location=DEVICE)
model.load_state_dict(ckpt['model'])
print(f"Loaded best model from epoch {ckpt['epoch']} (Val CER: {ckpt['val_cer']:.4f})")

test_cer, test_wer, test_preds, test_targets = evaluate(test_dl)
print(f'\n📊 Test Results')
print(f'   CER : {test_cer:.4f}  ({test_cer*100:.2f}%)')
print(f'   WER : {test_wer:.4f}  ({test_wer*100:.2f}%)')

In [ ]:
N_SHOW = 8
val_transform = get_val_transform()

fig, axes = plt.subplots(N_SHOW, 1, figsize=(16, N_SHOW * 1.8))
indices = random.sample(range(len(test_ds)), N_SHOW)

model.eval()
for ax, idx in zip(axes, indices):
    item   = dataset['test'][idx]
    img    = val_transform(item['image'].convert('RGB')).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(img)
    pred   = greedy_decode(logits)[0]
    target = item['text']

    char_err = cer([target], [pred])
    color    = 'green' if char_err < 0.05 else ('orange' if char_err < 0.2 else 'red')

    ax.imshow(item['image'], cmap='gray', aspect='auto')
    ax.set_title(
        f'GT: "{target}"  |  Pred: "{pred}"  |  CER: {char_err:.3f}',
        fontsize=9, color=color, pad=4
    )
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'qualitative_results.png'), dpi=150)
plt.show()

In [ ]:
from google.colab import files

def predict_image(image_path_or_pil):
    """Run inference on any handwritten text image."""
    if isinstance(image_path_or_pil, str):
        img = Image.open(image_path_or_pil)
    else:
        img = image_path_or_pil

    transform = get_val_transform()
    tensor    = transform(img.convert('RGB')).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logits = model(tensor)
    prediction = greedy_decode(logits)[0]

    plt.figure(figsize=(12, 2))
    plt.imshow(img, cmap='gray')
    plt.title(f'Prediction: "{prediction}"', fontsize=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    return prediction

# Upload your own image
print('Upload a handwritten text image:')
uploaded = files.upload()
for fname in uploaded:
    result = predict_image(fname)
    print(f'\n📝 Predicted text: "{result}"')

In [ ]:
# Cell 17 - Local OCR testing cell
# Provide image paths (or a glob pattern), then run this cell.

import glob
from pathlib import Path
from IPython.display import display

BEST_CKPT_PATH = os.path.join(SAVE_DIR, "htr_best.pt")


def load_best_model_for_testing():
    if not os.path.exists(BEST_CKPT_PATH):
        raise FileNotFoundError(f"Checkpoint not found: {BEST_CKPT_PATH}")

    ckpt = torch.load(BEST_CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    model.eval()
    print(f"Loaded checkpoint: {BEST_CKPT_PATH}")
    print(f"Epoch: {ckpt['epoch']} | Val CER: {ckpt['val_cer']:.4f}")


@torch.no_grad()
def predict_one(image_input):
    if isinstance(image_input, (str, Path)):
        image = Image.open(image_input).convert("RGB")
        image_name = str(image_input)
    elif isinstance(image_input, Image.Image):
        image = image_input.convert("RGB")
        image_name = "PIL_image"
    else:
        raise TypeError("image_input must be a path or PIL image")

    tensor = get_val_transform()(image).unsqueeze(0).to(DEVICE)
    logits = model(tensor)
    pred = greedy_decode(logits)[0]
    return image, image_name, pred


def run_ocr_test(image_inputs, max_show=6):
    records = []
    previews = []

    for item in image_inputs:
        try:
            image, image_name, pred = predict_one(item)
            records.append({"image": image_name, "prediction": pred})
            previews.append((image, image_name, pred))
        except Exception as exc:
            print(f"Skipping {item}: {exc}")

    if not records:
        print("No valid test images were processed.")
        return None

    results_df = pd.DataFrame(records)
    display(results_df)

    n_show = min(max_show, len(previews))
    fig, axes = plt.subplots(n_show, 1, figsize=(16, 2.2 * n_show))
    if n_show == 1:
        axes = [axes]

    for ax, (img, name, pred) in zip(axes, previews[:n_show]):
        ax.imshow(img, cmap="gray", aspect="auto")
        ax.set_title(f"{name}\nPrediction: \"{pred}\"", fontsize=10)
        ax.axis("off")

    plt.tight_layout()
    plt.show()
    return results_df


# -------------------- Usage --------------------
# Option 1: Explicit file list
TEST_IMAGES = [
    # r"C:\\path\\to\\line_01.png",
    # r"C:\\path\\to\\line_02.jpg",
]

# Option 2: Glob pattern (used only if TEST_IMAGES is empty)
TEST_GLOB = None
# Example: TEST_GLOB = r"C:\\path\\to\\test_lines\\*.png"

load_best_model_for_testing()

if TEST_IMAGES:
    selected_images = TEST_IMAGES
elif TEST_GLOB:
    selected_images = sorted(glob.glob(TEST_GLOB))
else:
    selected_images = []

if selected_images:
    _ = run_ocr_test(selected_images, max_show=6)
else:
    print("No test images provided. Add paths to TEST_IMAGES or set TEST_GLOB.")